# Lesson 3.1: Random Variables and Distributions

Build the probability distribution (PMF) of the sum of two dice as a table, find its mean and variance from the PMF, and check them by simulation.

Part of *Statistics from Scratch*. Run each cell with **Shift + Enter**, top to bottom.

## The question

A board game player rolls two fair dice and moves forward by their total. She wants to know the whole picture: how likely is each total from 2 to 12, how likely is a total of 4 or less, and what is the average total and how spread out is it? Can we build the probability distribution as a table and then check it by simulation?

If a package is missing on your computer, run the next cell once. In Google Colab everything is already installed.

In [ ]:
%pip install -q numpy scipy pandas statsmodels matplotlib

## Step 1: The PMF as a table

We count how many of the 36 equally likely pairs give each sum, then divide by 36. A pandas table keeps the values, the counts and the probabilities side by side.

In [1]:
import numpy as np  # Load NumPy for numbers.
import pandas as pd  # Load pandas to hold the table.
from itertools import product  # Load product to list all the dice pairs.

sums = [a + b for a, b in product(range(1, 7), repeat=2)]  # Make the total of every one of the 36 equally likely pairs.
pmf = pd.Series(sums).value_counts().sort_index() / 36  # Count how often each total occurs, sort the totals, and divide by 36. This is the probability mass function (PMF).

table = pd.DataFrame({'ways': (pmf * 36).astype(int), 'P(X = x)': pmf.round(4)})  # Put the ways to get each total and its probability side by side in one table.
table.index.name = 'sum x'  # Name the row labels 'sum x' so the table is easy to read.
print(table.to_string())  # Print the table in full.

       ways  P(X = x)
sum x                
2         1    0.0278
3         2    0.0556
4         3    0.0833
5         4    0.1111
6         5    0.1389
7         6    0.1667
8         5    0.1389
9         4    0.1111
10        3    0.0833
11        2    0.0556
12        1    0.0278


**What this means.** The table gives 1 way to get a 2, 2 ways for a 3, rising to 6 ways for a 7 (probability 0.1667) and then falling back to 1 way for a 12. The middle totals are the most likely and the extremes the rarest.

## Step 2: Check it is a valid PMF and add the CDF

A valid PMF has probabilities between 0 and 1 that add to exactly 1. The running total (cumsum) is the CDF, which answers 'at most' questions at once.

In [2]:
print('total probability:', round(pmf.sum(), 10))  # Add up all the probabilities. A valid PMF must total exactly 1.

cdf = pmf.cumsum()  # Running total of the probabilities. This is the cumulative distribution function (CDF).
print(f'P(X <= 4)  = {cdf[4]:.4f}')  # Read the CDF at 4: the chance the total is at most 4.
print(f'P(X >= 10) = {1 - cdf[9]:.4f}')  # For 10 or more, take 1 minus the CDF at 9, because 'at least 10' is everything except 9 or less.

total probability: 1.0
P(X <= 4)  = 0.1667
P(X >= 10) = 0.1667


**What this means.** The probabilities add to 1.0, so this is a valid distribution. P(X <= 4) = 0.1667 and P(X >= 10) = 0.1667. They match because the distribution is symmetric around 7.

## Step 3: Mean and variance from the PMF

The mean is the sum of x times P(x). The variance is the sum of (x minus the mean) squared times P(x). These describe the whole population of possible rolls, so there is no n - 1 here.

In [3]:
x = pmf.index.to_numpy()  # Pull out the possible totals as an array.
p = pmf.to_numpy()  # Pull out their probabilities as an array.

mean = (x * p).sum()  # The mean is the sum of each total times its probability.
var = ((x - mean) ** 2 * p).sum()  # The variance is the sum of squared gaps from the mean, each weighted by its probability.

print(f'mean     = {mean:.4f}')  # Print the mean.
print(f'variance = {var:.4f}')  # Print the variance.
print(f'sd       = {var ** 0.5:.4f}')  # Print the standard deviation, the square root of the variance, back in the original units.

mean     = 7.0000
variance = 5.8333
sd       = 2.4152


**What this means.** The mean total is 7.0000, the variance is 5.8333 and the standard deviation is 2.4152. A typical roll lands within about 2.4 of 7. We divide by probabilities, not by n - 1, because this is the whole population of possible rolls.

## Step 4: Check by simulation

Roll two dice 100,000 times with a seeded generator. The simulated mean and variance should sit close to the exact ones. ddof=0 matches the population formula above.

In [4]:
rng = np.random.default_rng(42)  # Create a seeded generator so the simulation repeats.
rolls = rng.integers(1, 7, size=100_000) + rng.integers(1, 7, size=100_000)  # Roll two dice 100,000 times and add them.

print(f'simulated mean     = {rolls.mean():.4f}')  # Print the simulated mean, which should be close to 7.
print(f'simulated variance = {rolls.var(ddof=0):.4f}')  # Print the simulated variance with ddof=0, the same population formula as above.
print(f'simulated P(X = 7) = {(rolls == 7).mean():.4f}  (exact {pmf[7]:.4f})')  # Print the simulated chance of a 7 next to the exact one.

simulated mean     = 7.0055
simulated variance = 5.7999
simulated P(X = 7) = 0.1688  (exact 0.1667)


**What this means.** The simulation gave a mean of 7.0055, a variance of 5.7999 and P(X = 7) = 0.1688, against the exact 7, 5.8333 and 0.1667. They are close but not identical because simulations have a little random error.

## So, what is the answer?

The total of two dice has a mean of 7, a variance of 5.8333 and an SD of 2.4152, and totals near 7 are the most likely (a 7 has probability 0.1667, a 2 or a 12 only 0.0278). Totals of 4 or less and of 10 or more each have a probability of 0.1667. The caution: the simulation matched only approximately, and these formulas assume the dice are fair and independent.

## Now you try

A shop records how many items a customer buys. After a long study the manager believes the chances are as listed below. She wants to know the average basket size, how variable it is, and how likely a customer is to buy 3 or more items.

**Data:** Items bought: x = [0, 1, 2, 3, 4] with probabilities p = [0.10, 0.30, 0.35, 0.15, 0.10]. For the simulation use the seed 11 and 100,000 customers.

1. Check that the probabilities add up to 1.
2. Find the mean, variance and standard deviation of the number of items.
3. Find P(X >= 3).
4. Simulate 100,000 customers and compare the mean and variance.

*Hint:* The mean is the sum of x times p, and the variance is the sum of (x - mean) squared times p. For the simulation use rng.choice(x, size=100_000, p=p).

In [ ]:
# Your turn: write your code here


### Solution (try first, then peek)

In [ ]:
import numpy as np

x = np.array([0, 1, 2, 3, 4])
p = np.array([0.10, 0.30, 0.35, 0.15, 0.10])
print('total probability:', round(p.sum(), 10))

mean = (x * p).sum()
var = ((x - mean) ** 2 * p).sum()
print(f'mean = {mean:.3f}, variance = {var:.4f}, sd = {var ** 0.5:.4f}')
print(f'P(X >= 3) = {p[x >= 3].sum():.2f}')

rng = np.random.default_rng(11)
sim = rng.choice(x, size=100_000, p=p)
print(f'simulated mean = {sim.mean():.3f}, variance = {sim.var():.4f}')

total probability: 1.0
mean = 1.850, variance = 1.2275, sd = 1.1079
P(X >= 3) = 0.25
simulated mean = 1.850, variance = 1.2262


**Answer.** The probabilities add to 1.0. The mean is 1.850 items, the variance 1.2275 and the SD 1.1079, so the typical basket is within about 1.1 items of 1.85. P(X >= 3) = 0.15 + 0.10 = 0.25, so one customer in four buys 3 or more. The simulation gave a mean of 1.850 and a variance of 1.2262, matching the exact values closely.

## The same question in R, SPSS and Stata

**R**

```r
s <- as.vector(outer(1:6, 1:6, "+"))  # List the 36 sums with outer(), which adds every die 1 value to every die 2 value.
pmf <- table(s) / 36  # Count each sum with table() and divide by 36 to get the PMF.
x <- as.numeric(names(pmf))  # Get the sums themselves as numbers.
round(pmf, 4)  # Print the PMF rounded to four decimals.
mu <- sum(x * pmf)  # The mean: sum of x times P(x).
mu  # Print the mean.
sum((x - mu)^2 * pmf)  # Variance: weighted squared gaps from the mean.
sqrt(sum((x - mu)^2 * pmf))  # Standard deviation: the square root.

set.seed(42)  # Fix the seed.
rolls <- sample(6, 1e5, TRUE) + sample(6, 1e5, TRUE)  # Simulate 100,000 rolls of two dice.
c(mean(rolls), mean((rolls - mean(rolls))^2))  # Report the simulated mean and the simulated population variance.
```

How to read it: The printed PMF has 0.1667 for a sum of 7. Then come the mean 7, the variance 5.8333 and the standard deviation 2.4152. The simulated pair at the end should be close to 7 and 5.83, not identical.

**SPSS** (Analyze > Descriptive Statistics > Frequencies (Statistics: Mean, Variance))

```
INPUT PROGRAM.
LOOP #a = 1 TO 6.
LOOP #b = 1 TO 6.
COMPUTE total = #a + #b.
END CASE.
END LOOP.
END LOOP.
END FILE.
END INPUT PROGRAM.
FREQUENCIES VARIABLES=total
  /STATISTICS=MEAN VARIANCE STDDEV.
```

How to read it: In the Frequencies table, the Percent column is the PMF (16.7 for a total of 7) and Cumulative Percent is the CDF. In Statistics the Mean is 7.00. SPSS reports Variance 6.000 because it divides by n - 1 = 35.

**Stata**

```
clear
set obs 36
gen total = ceil(_n/6) + mod(_n - 1, 6) + 1
tabulate total
summarize total
display r(Var) * 35/36

clear
set seed 42
set obs 100000
gen rolls = ceil(6*runiform()) + ceil(6*runiform())
summarize rolls
```

How to read it: In the tabulate output, read the Percent column as the PMF (16.67 for 7) and Cum. as the CDF. summarize shows Mean 7, and the display line prints 5.8333, the population variance. The simulation's mean is near 7 but not identical to Python's.